In [3]:
'''
How to use:
1-Locate the input_data.txt file in the folder.
2-Enter one or more transactions in input_data.txt, one per line.
3-Data must be formatted as <step>,<type>,<amount>,<nameOrig>,<oldbalanceOrg>,<newbalanceOrig>,<nameDest>,<oldbalanceDest>,<newbalanceDest>,<isFlaggedFraud>
4-Run the notebook or the analyzer flow.
5-Open output_analysis.txt to see the model prediction, an explanation, and an LLM-generated summary for each transaction.
'''
import os
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests

model_path = Path.cwd() / "../code/custom-classifier-model/fraud_detector.pkl"
model_path = model_path.resolve()
artifact = joblib.load(model_path)

if isinstance(artifact, dict) and "pipeline" in artifact:
    pipeline = artifact["pipeline"]
    threshold = float(artifact.get("threshold", 0.5))
    feature_columns = artifact.get("feature_columns", [])
else:
    pipeline = artifact
    threshold = 0.5
    feature_columns = []

if feature_columns:
    feature_columns = list(feature_columns)

print(f"Loaded model artifact from: {model_path}")
print(f"Threshold = {threshold}")
print(f"Feature columns = {feature_columns}")


Loaded model artifact from: C:\Users\mbaoj\CSCE581-Spring2025-MatthewBojanowski\code\custom-classifier-model\fraud_detector.pkl
Threshold = 0.2496513755307867
Feature columns = ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'nameOrig_prefix', 'nameDest_prefix', 'orig_balance_delta', 'dest_balance_delta', 'amount_to_oldOrg', 'amount_to_newOrg']


In [4]:
def prepare_features(df):
    df = df.copy()
    df['type'] = df['type'].astype(str)
    df['nameOrig'] = df['nameOrig'].astype(str)
    df['nameDest'] = df['nameDest'].astype(str)

    df['nameOrig_prefix'] = df['nameOrig'].str[0]
    df['nameDest_prefix'] = df['nameDest'].str[0]
    df['orig_balance_delta'] = df['newbalanceOrig'] - df['oldbalanceOrg']
    df['dest_balance_delta'] = df['newbalanceDest'] - df['oldbalanceDest']

    old_org_denom = df['oldbalanceOrg'].replace(0, np.nan).fillna(1)
    new_org_denom = df['newbalanceOrig'].replace(0, np.nan).fillna(1)
    df['amount_to_oldOrg'] = df['amount'] / old_org_denom
    df['amount_to_newOrg'] = df['amount'] / new_org_denom

    if feature_columns:
        return df[feature_columns].copy()
    return df


def build_local_reason(row, probability):
    signals = []
    if row['type'].lower() in {'transfer', 'cash_out'}:
        signals.append(f"transaction type '{row['type']}' is riskier than a PAYMENT")
    if row['amount_to_oldOrg'] > 10:
        signals.append("amount is large relative to the origin's previous balance")
    if row['amount_to_newOrg'] > 10:
        signals.append("amount is large relative to the origin's new balance")
    if row['orig_balance_delta'] < 0:
        signals.append("origin balance dropped after the transaction")
    if row['dest_balance_delta'] == 0:
        signals.append("destination balance did not increase")
    if not signals:
        signals.append("no obvious high-risk signal found in the engineered features")

    return f"Model probability={probability:.6f}. " + " ".join(signals)


def get_llm_explanation(transaction, prediction, probability, threshold, local_reason):
    prompt = (
        "You are a financial fraud analyst. "
        "A machine learning fraud model analyzed a single transaction and produced a prediction. "
        "Explain why the model classified the transaction as fraud or not fraud in clear, non-technical language. "
        "Include the key factors from the transaction and the model output.\n\n"
        f"Transaction: {json.dumps(transaction, default=str)}\n"
        f"Predicted fraud: {bool(prediction)}\n"
        f"Fraud probability: {probability:.6f}\n"
        f"Threshold: {threshold:.6f}\n"
        f"Local explanation: {local_reason}\n"
        "If an LLM service is not available, return a short text that clearly states this and summarizes the decision."
    )

    api_key = os.environ.get('OPENAI_API_KEY')
    api_base = os.environ.get('OPENAI_API_BASE')
    model_name = os.environ.get('OPENAI_MODEL', 'gpt-3.5-turbo')

    if api_key:
        try:
            import openai
            openai.api_key = api_key
            if api_base:
                openai.api_base = api_base
            completion = openai.ChatCompletion.create(
                model=model_name,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0.0,
                max_tokens=250,
            )
            return completion.choices[0].message.content.strip()
        except Exception:
            try:
                url = (api_base or 'https://api.openai.com/v1').rstrip('/') + '/chat/completions'
                response = requests.post(
                    url,
                    headers={
                        'Authorization': f'Bearer {api_key}',
                        'Content-Type': 'application/json',
                    },
                    json={
                        'model': model_name,
                        'messages': [{'role': 'user', 'content': prompt}],
                        'temperature': 0.0,
                        'max_tokens': 250,
                    },
                    timeout=30,
                )
                response.raise_for_status()
                data = response.json()
                return data['choices'][0]['message']['content'].strip()
            except Exception as ex:
                return (
                    "LLM request failed while trying to explain the transaction. "
                    "Using the local reason instead: " + local_reason
                )

    return (
        "LLM unavailable because OPENAI_API_KEY is not set. "
        "Using local model output instead: " + local_reason
    )


def analyze_data(user_input_data):
    if isinstance(user_input_data, dict):
        if not any(isinstance(v, (list, tuple, pd.Series)) for v in user_input_data.values()):
            user_df = pd.DataFrame([user_input_data])
        else:
            user_df = pd.DataFrame(user_input_data)
    elif isinstance(user_input_data, pd.Series):
        user_df = user_input_data.to_frame().T
    else:
        user_df = pd.DataFrame(user_input_data)

    expected_columns = [
        'step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg',
        'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud'
    ]
    user_df = user_df[expected_columns]
    features = prepare_features(user_df)
    probabilities = pipeline.predict_proba(features)[:, 1]
    predictions = (probabilities >= threshold).astype(int)

    results = []
    for idx, row in user_df.iterrows():
        prob = float(probabilities[idx])
        pred = int(predictions[idx])
        feature_row = prepare_features(pd.DataFrame([row])).iloc[0]
        local_reason = build_local_reason(feature_row, prob)
        llm_text = get_llm_explanation(row.to_dict(), pred, prob, threshold, local_reason)
        conclusion = 'Fraud detected' if pred == 1 else 'No fraud detected'
        results.append({
            'step': row['step'],
            'type': row['type'],
            'amount': row['amount'],
            'nameOrig': row['nameOrig'],
            'oldbalanceOrg': row['oldbalanceOrg'],
            'newbalanceOrig': row['newbalanceOrig'],
            'nameDest': row['nameDest'],
            'oldbalanceDest': row['oldbalanceDest'],
            'newbalanceDest': row['newbalanceDest'],
            'isFlaggedFraud': row['isFlaggedFraud'],
            'prediction': pred,
            'fraud_probability': prob,
            'threshold': threshold,
            'reason': local_reason,
            'llm_explanation': llm_text,
            'conclusion': conclusion,
        })

    return pd.DataFrame(results)


In [ ]:
# example user input matching the models expected features
user_input_data = {
    'step': 1,
    'type': 'PAYMENT',
    'amount': 1000.0,
    'nameOrig': 'C123456789',
    'oldbalanceOrg': 5000.0,
    'newbalanceOrig': 4000.0,
    'nameDest': 'M123456789',
    'oldbalanceDest': 0.0,
    'newbalanceDest': 0.0,
    'isFlaggedFraud': 0,
}

# call the analyze_data function
analyze_data(user_input_data)

In [ ]:
# simple test cases
# not fraud
non_fraud_tx = {
    'step': 1,
    'type': 'PAYMENT',
    'amount': 100.0,
    'nameOrig': 'C000000001',
    'oldbalanceOrg': 1000.0,
    'newbalanceOrig': 900.0,
    'nameDest': 'M000000001',
    'oldbalanceDest': 0.0,
    'newbalanceDest': 0.0,
    'isFlaggedFraud': 0,
}
# call the analyze_data function
analyze_data(non_fraud_tx)

In [1]:
fraud_like_tx = {
    'step': 1,
    'type': 'TRANSFER',
    'amount': 250000.0,
    'nameOrig': 'C000000002',
    'oldbalanceOrg': 250000.0,
    'newbalanceOrig': 0.0,
    'nameDest': 'C000000003',
    'oldbalanceDest': 0.0,
    'newbalanceDest': 0.0,
    'isFlaggedFraud': 0,
}
# call the analyze_data function
analyze_data(fraud_like_tx)

In [ ]:
fraud_like_tx2 = {
    'step': 1,
    'type': 'TRANSFER',
    'amount': 77777777777.00,
    'nameOrig': '67',
    'oldbalanceOrg': 0.0,
    'newbalanceOrig': 77777777777.00,
    'nameDest': '68',
    'oldbalanceDest': 77777777777.0,
    'newbalanceDest': 0.0,
    'isFlaggedFraud': 0,
}
analyze_data(fraud_like_tx2)

In [5]:
def load_input_data(filepath):
    columns = [
        'step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg',
        'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud'
    ]
    return pd.read_csv(
        filepath,
        header=None,
        names=columns,
        dtype={
            'step': int,
            'type': str,
            'amount': float,
            'nameOrig': str,
            'oldbalanceOrg': float,
            'newbalanceOrig': float,
            'nameDest': str,
            'oldbalanceDest': float,
            'newbalanceDest': float,
            'isFlaggedFraud': int,
        },
        on_bad_lines='skip',
    )


def write_output_analysis(result_df, output_path):
    result_df = result_df.copy()
    csv_text = result_df.drop(columns=['llm_explanation']).to_csv(index=False)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(csv_text)
        f.write('\n\nLLM Explanations:\n')
        for idx, row in result_df.iterrows():
            f.write(f"Line {idx + 1}: {row['llm_explanation']}\n\n")

    print(f"Wrote analysis to {output_path}")


input_filepath = 'input_data.txt'
output_filepath = 'output_analysis.txt'

input_data = load_input_data(input_filepath)
results = analyze_data(input_data)
write_output_analysis(results, output_filepath)

results.head()


Wrote analysis to output_analysis.txt


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFlaggedFraud,prediction,fraud_probability,threshold,reason,llm_explanation,conclusion
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,0.000000,0.249651,Model probability=0.000000. origin balance dro...,LLM unavailable because OPENAI_API_KEY is not ...,No fraud detected
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,0.000000,0.249651,Model probability=0.000000. origin balance dro...,LLM unavailable because OPENAI_API_KEY is not ...,No fraud detected
2,1,TRANSFER,182.00,C1305486145,182.0,0.00,C553264065,0.0,0.0,0,1,0.992424,0.249651,Model probability=0.992424. transaction type '...,LLM unavailable because OPENAI_API_KEY is not ...,Fraud detected
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,0,1,0.917289,0.249651,Model probability=0.917289. transaction type '...,LLM unavailable because OPENAI_API_KEY is not ...,Fraud detected
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,0.000000,0.249651,Model probability=0.000000. origin balance dro...,LLM unavailable because OPENAI_API_KEY is not ...,No fraud detected
